
##### Silver Notebook #####

**Utilities**

In [1]:
notebookutils.fs.help()

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 3, Finished, Available, Finished, False)

In [2]:
 notebookutils.fs.ls("Files")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://385167aa-eb1b-47d3-ac23-ce57420d5875@onelake.dfs.fabric.microsoft.com/2492b3e8-b1dd-47fc-a221-adacf43aafa8/Files/Store_data, name=Store_data, size=0)]

**Data Ingestion**

_From Same Lakehouse_

In [3]:
df = spark.read.format("csv")\
            .option("header", True)\
            .option("inferSchema", True)\
            .load("Files/Store_data")

display(df)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76e61f03-0b9c-480b-8448-f36864dbcda7)

_From different (Bronze) lakehouse_

In [4]:
df_cust = spark.read.format("Delta")\
                .load("abfss://Fabric_Project_WS@onelake.dfs.fabric.microsoft.com/Bronze_LH_DE.Lakehouse/Tables/raw/customer_dim")

display(df_cust)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee1736a3-2719-4160-87ef-ed3480604946)

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 7, Finished, Available, Finished, False)

In [9]:
df_cust = df_cust.withColumn("first_name", split(col('name'), ' ')[0])\
                 .withColumn("last_name", coalesce(split(col('name'), ' ')[1]))

display(df_cust)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 511b2377-c99e-49c2-a54c-1f1a1463c91f)

In [12]:
df_cust = df_cust.fillna({'last_name':'NA'})
display(df_cust)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bee0c10f-088f-4258-b913-583c5f4054ba)

In [26]:
df_cust.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable("Silver_LH_DE.raw.Customer_dim")


StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 28, Finished, Available, Finished, False)

In [27]:
df_fact = spark.read.table("Bronze_LH_DE.raw.fact_table")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 29, Finished, Available, Finished, False)

In [29]:
display(df_fact)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54761085-9670-4a4a-b552-893ca97f3707)

In [34]:
df_fact = df_fact.withColumn("quantity", col("quantity").cast(IntegerType()))\
                .withColumn("unit_price", col("unit_price").cast(IntegerType()))\
                .withColumn("total_price", col("total_price").cast(IntegerType()))

df_fact.printSchema()
display(df_fact)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 36, Finished, Available, Finished, False)

root
 |-- payment_key: string (nullable = true)
 |-- coustomer_key: string (nullable = true)
 |-- time_key: string (nullable = true)
 |-- item_key: string (nullable = true)
 |-- store_key: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- total_price: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, f8ef427f-9c3a-4076-b9fb-4e59177b5ace)

In [45]:
df_fact.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable("Silver_LH_DE.raw.fact_table")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 47, Finished, Available, Finished, False)

In [35]:
df_store = spark.read.table("Bronze_LH_DE.raw.store_dim")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 37, Finished, Available, Finished, False)

In [38]:
df_store = df_store.withColumn("Address", concat(col("district"),lit("-"),col("upazila")))

display(df_store)

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1c64843-e095-4131-9ada-8963f6d5d6ef)

In [40]:
df_store = df_store.drop("district", "upazila")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 42, Finished, Available, Finished, False)

In [41]:
df_store.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable("Silver_LH_DE.raw.store_dim")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 43, Finished, Available, Finished, False)

In [42]:
df_trans = spark.read.table("Bronze_LH_DE.raw.trans_dim")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 44, Finished, Available, Finished, False)

In [43]:
df_trans = df_trans.withColumn("bank_name", regexp_replace(col("bank_name"), "None", "Not Available"))

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 45, Finished, Available, Finished, False)

In [44]:
df_trans.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable("Silver_LH_DE.raw.trans_dim")

StatementMeta(, 553b3725-fa65-47e1-9ed8-099a543a1367, 46, Finished, Available, Finished, False)